## Claude.ai

Push VQA to Claude.

### Docs:

* in theory, there is documentation here: https://docs.anthropic.com/en/home, 
* but in reality I just asked "Hey Claude.  How do I use your API through Python to upload an image and ask you questions about it?" followed by "Is there anyway to set the context?  For example, in chatgpt you have the ""role": "system", "content"" part of your message where you would say things like "You are a helpful assistant in charge of automating a process".  Or does one just incorporate that into the "question" part of the inputs?" and starting building from there

#### Key Points (cp-claude)

* API Key: Get your API key from the [Anthropic Console](https://console.anthropic.com/) and set it as an environment variable or pass it directly
* Supported formats: JPEG, PNG, GIF, and WebP images
* Model: Use `claude-sonnet-4-20250514` for Claude Sonnet 4
* Size limits: Images should be under 5MB and no larger than 8000x8000 pixels



First, install anthropic api (also, see .yml file for the environment for this project)

In [17]:
#!pip install anthropic

Where are things stored/going to be stored?

In [18]:
# dir_api = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku/' #store API results for example data
# model = "claude-haiku-4-5-20251001" # nano-like equivalent
# thinking = False # not supported for claude
# max_tokens = None

# dir_api = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_sonnet/' #store API results for example data
# model = "claude-sonnet-4-6" # nano-like equivalent
# thinking = True # not supported for claude
# max_tokens = None

# dir_api = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku_thinking/' #store API results for example data
# model = "claude-haiku-4-5-20251001" # nano-like equivalent
# thinking = True 
# max_tokens = None

dir_api = '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku_thinking_maxT8000/' #store API results for example data
model = "claude-haiku-4-5-20251001" # nano-like equivalent
thinking = True 
max_tokens = 10000 # 8000(thinking) + 2000(response)

# where is VQA dataset?
jsons_dir = '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/' # directory where jsons created with figure are stored
imgs_dir = '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/' # where images are stored

# other stuff, where stored?
key_file = '/Users/jnaiman/.claudeai/key.txt'
# for saving temp images for reading in
tmp_dir = '/Users/jnaiman/Downloads/tmp/'

img_format = 'jpeg'

# for asking for reasoning
reasoning_text = 'In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.'

In [19]:
import anthropic
import base64
from PIL import Image
import numpy as np
import json
import re
import pickle
import os
from glob import glob

# debug
from importlib import reload
from anthropic import RateLimitError


from sys import path
path.append('../')
import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import parse_qa, load_image, get_img_json_pair, parse_for_errors

import time
from utils.plot_qa_utils import get_nplots

In [20]:
# setup
with open(key_file,'r') as f:
    api_key = f.read()

client = anthropic.Anthropic(
  api_key=api_key.strip(),  # this is also the default, it can be omitted
)

In [21]:
jsons_to_parse = glob(jsons_dir + '/*.json')
jsons_to_parse[:3]

['/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000015_qa.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000005_qa.json',
 '/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/qa_jsons/Picture_000077_qa.json']

In [22]:
def send_to_claude(question_list, client, image_path, encoded_image,
                    model="claude-sonnet-4-20250514",
                    max_tokens=1000, temperature=0.1,
                    #media_type = 'image/', ## JPN take this out!
                    tmp_dir = '/Users/jnaiman/Downloads/tmp/',
                    test_run = True, fac=1.0, 
                    verbose=True, verbose_tokens = False,
                    system_prompt = None, 
                    max_retries = 10, sleep_time=1, 
                    img_format='jpeg', reasoning=None, thinking = True, 
                    thinking_tokens=8000):
    """
    Sends the question to claude and collects response.clear

    system_prompt : if None, then defaults to the overall system prompt generated with the questions.
    """
    if system_prompt is None:
        system_prompt = question_list['persona']
    # update for caching
    system_prompt = [{
        "type": "text",
        "text": system_prompt,
        "cache_control": {"type": "ephemeral"}
    }]

    media_type = 'image/' + img_format #+= img_format
    #print('media type:', media_type)

    #print('system prompt:', system_prompt)

    iFac = 2.0
    success = False
    #['persona', 'context','question', 'format']
    attempt = 0
    while not success and attempt < max_retries:
        try:
            question = question_list['context'] + " " + question_list['question'] + " " + question_list['format']
            if reasoning is not None:
                question += " " + reasoning
            # lowercase the first word, just in case
            question = question.lstrip() # no whitespace
            question = question[0].lower() + question[1:]
            if verbose: print('   on question:',question)
            # Prepare the API request
            prompt = f"I am going to show you an image. Now, {question}"
            prompt_save = f"I am going to show you an image. Now, {question}"
            ##question_list['prompt'] = prompt
            
            if not test_run:
                if thinking and 'haiku' not in model:
                    # Send the request to the Claude api
                    response = client.messages.create(
                        model = model,
                        max_tokens=max_tokens,
                        system = system_prompt,
                        #temperature=temperature,
                        thinking={"type": "adaptive"},  # <-- top-level parameter
                        messages=[
                            {
                                "role": "user",
                                "content": [
                                    {
                                        "type": "image",
                                        "source": {
                                            "type": "base64",
                                            "media_type": media_type,
                                            "data": encoded_image,
                                        },
                                    },
                                    {
                                        "type": "text",
                                        "text": prompt
                                    }
                                ],
                            }
                        ]
                    )
                elif thinking and 'haiku' in model:
                    # Send the request to the Claude api
                    if thinking_tokens is not None:
                        #thinking_tokens = max(min(thinking_tokens, max_tokens//2), 1024)
                        thinking_tokens = max(max_tokens-2000, 1024)
                    response = client.messages.create(
                        model = model,
                        max_tokens=max_tokens,
                        system = system_prompt,
                        #temperature=temperature,
                        thinking={
                                "type": "enabled",
                                "budget_tokens": thinking_tokens   # min 1024, up to 64k for Haiku
                            },                        
                            messages=[
                            {
                                "role": "user",
                                "content": [
                                    {
                                        "type": "image",
                                        "source": {
                                            "type": "base64",
                                            "media_type": media_type,
                                            "data": encoded_image,
                                        },
                                    },
                                    {
                                        "type": "text",
                                        "text": prompt
                                    }
                                ],
                            }
                        ]
                    )
                else:
                    # Send the request to the Claude api
                    response = client.messages.create(
                        model = model,
                        max_tokens=max_tokens,
                        system = system_prompt,
                        temperature=temperature,
                        #thinking={"type": "adaptive"},  # <-- top-level parameter
                        messages=[
                            {
                                "role": "user",
                                "content": [
                                    {
                                        "type": "image",
                                        "source": {
                                            "type": "base64",
                                            "media_type": media_type,
                                            "data": encoded_image,
                                        },
                                    },
                                    {
                                        "type": "text",
                                        "text": prompt
                                    }
                                ],
                            }
                        ]
                    )
                success = True
            else:
                success = True
        except RateLimitError as e:
            if attempt < max_retries - 1:
                # Exponential backoff with jitter
                wait_time = sleep_time*(2 ** (attempt)) + np.random.uniform(0, 1)
                print(f"      Rate limit hit. Waiting {wait_time:.2f} seconds before retry {attempt + 1}")
                time.sleep(wait_time)
                attempt += 1
            else:
                print(f"Max retries exceeded. Error: {e}")
                raise
        except Exception as e:
            print(e)
            new_fac = fac/iFac
            print('      new fac = ', new_fac)
            encoded_image = load_image(image_path,fac=new_fac, tmp_dir=tmp_dir)
            iFac += 1
    
    #print(response)
    if not test_run:
        # Get the response from the API
        #answer = response.content[0].text #response.choices[0].message.content
        # try:
        #     answer = next(b.text for b in response.content if b.type == "text")
        # except:
        #     print(response.content)
        #     print([b.type for b in response.content])
        #     lsklsj
        text_blocks = [b for b in response.content if b.type == "text"]
        if text_blocks:
            answer = text_blocks[0].text
        else:
            print("WARNING: no text block found, content was:", response.content)
            print([b.type for b in response.content])
            answer = ""
            question_list['Error'] = 'No text block in response'
        question_list['raw answer'] = answer
        # also calculate usage
        usage = response.usage
        question_list['usage'] = usage
        if verbose and verbose_tokens:
            print(f"      - Input tokens: {usage.input_tokens}")
            print(f"      - Output tokens: {usage.output_tokens}")
            print(f"      - Total tokens: {usage.input_tokens + usage.output_tokens}")
        # format answer
        answer_format = answer.split('```json"')[-1].split('\n')[0].replace('\n', '')
        #answer.replace("```json\n",'').replace("\n```",'')
        try:
            question_list['Response'] = json.loads(answer_format)
        except:
            question_list['Response'] = answer_format
            question_list['Error'] = 'JSON formatting'
        question_list['Response String'] = answer_format
        success = True
    else:
        question_list['Response'] = 'TEST RUN'
        question_list['Response String'] = 'TEST RUN'
        question_list['raw answer'] = 'TEST RUN'

    return question_list, prompt_save, system_prompt

In [23]:
iMax = 1000
verbose = False
test_run = False # run w/o actually pinging openai
restart = False
#max_tokens=500
if max_tokens is None:
    max_tokens = 8000
#max_tokens=10000 
#max_tokens=1000 

# set system_prompt to None to default to what is in question list
system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any explanations, reasoning, or text outside of the JSON response."""
#system_prompt = """You must respond with only valid JSON. Start your response immediately with { and end with }. Do not write any text before or after the JSON."""
if reasoning_text is not None: # rephrease is reasoning is requested
    system_prompt = """You are a helpful assistant that responds only in valid JSON format. Do not include any text outside of the requested JSON responses."""

temperature=0.1
fac = 1.0
sleep = 5 # seconds
use_single_prompt = True # use 1 prompt and 1 image, if False will use multiple at a time

max_img_size = (8000,8000)

import utils.llm_utils
reload(utils.llm_utils)
from utils.llm_utils import get_img_json_pair

if test_run:
    sleep = 0.0

for ijson,json_path in enumerate(jsons_to_parse):
    if ijson >= iMax:
        continue

    print('on', ijson, 'of', min(iMax,len(jsons_to_parse)))

    # get image and base json
    img_path = imgs_dir + json_path.split('/')[-1].removesuffix('_qa.json') + '.' + img_format
    print(img_path)
    encoded_image, img_format_media, base_json, err = get_img_json_pair(img_path, json_path, dir_api, 
                                                      fac=fac, restart=restart, img_format=img_format,
                                                      tmp_dir=tmp_dir, max_img_size=max_img_size)
    if err:
        continue

    ###### create QA ########
    qa = []
    
    for k,v in base_json['VQA']['Level 1']['Figure-level questions'].items():
        out = {'Q':v['Q'], 'A':v['A'], 'Level':'Level 1', 'type':'Figure-level questions', 'Response':"", 
               "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
               "reasoning":reasoning_text}
        qa.append(out)
    for level in ['Level 2', 'Level 3']:
        if level in base_json['VQA']:
            if 'Figure-level questions' in base_json['VQA'][level]:
                #print('** yes, level ***', level)
                for k,v in base_json['VQA'][level]['Figure-level questions'].items():
                    out = {'Q':v['Q'], 'A':v['A'], 'Level':level, 'type':'Figure-level questions', 'Response':"", 
                        "persona":v['persona'], 'context':v['context'], 'question':v['question'], 'format':v['format'],
                        "reasoning":reasoning_text}
                    qa.append(out)
    
    # what kinds?
    #types = ['(words + list)', '(words)']
    types = []
    
    # get uniques
    level_parse = 'Level 1'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types, use_split_keys=False)
    
    level_parse = 'Level 2'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types, use_split_keys=False)
    
    level_parse = 'Level 3'
    plot_level = 'Plot-level questions'
    qa = parse_qa(level_parse, plot_level, qa, base_json['VQA'], types, use_split_keys=False)

    responses = []; prompts = []; system_prompts = []
    if use_single_prompt:
        for question_list in qa:
            response, prompt, system_prompt_out = send_to_claude(question_list, client, img_path, encoded_image,
                        model = model, max_tokens=max_tokens, #media_type='image/' + img_format_media,
                        test_run = test_run, system_prompt=system_prompt, temperature=temperature, 
                        sleep_time=sleep, fac=fac, img_format=img_format, reasoning = reasoning_text, thinking=thinking)
            responses.append(response)
            question_list['prompt'] = prompt
            question_list['system prompt'] = system_prompt_out
        #import sys; sys.exit() # just do one
        time.sleep(sleep)
    time.sleep(sleep)

    # parse for errors
    #qa = parse_for_errors(qa, llm='claude')
    print('')
    print('**** Cleaned QA ****')
    #qa = parse_for_errors_claude(qa)
    qa = parse_for_errors(qa, llm='Claude')

    # dump to file
    if not test_run:
        with open(dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle', 'wb') as ff:
            pickle.dump([qa, model], ff)
        print('Just saved:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    else:
        print('Would store at:', dir_api + json_path.split('/')[-1].removesuffix('.json')+ '.pickle')
    #import sys; sys.exit()
print("!!!!!!!! DONE !!!!!!!!!!!")

on 0 of 200
/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/Picture_000015.jpeg
have file already: /Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku_thinking_maxT8000/Picture_000015_qa.pickle
on 1 of 200
/Users/jnaiman/Dropbox/jcdl_followup/synthetic_figures/imgs/Picture_000005.jpeg
   on question: how many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
   on question: assume this is a figure made with matplotlib in Python.  Examples of plotting styles are "classic" or "ggplot". Examples of plotting styles are "classic" or "ggplot". What is the plot style used in this figure? Please format the output as 

## Look at data

Check out one, if you wanna:

In [24]:
pickles = glob(dir_api + '*.pickle')
pickles[:5]

['/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku_thinking_maxT8000/Picture_000193_qa.pickle',
 '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku_thinking_maxT8000/Picture_000066_qa.pickle',
 '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku_thinking_maxT8000/Picture_000120_qa.pickle',
 '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku_thinking_maxT8000/Picture_000134_qa.pickle',
 '/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku_thinking_maxT8000/Picture_000072_qa.pickle']

In [25]:
ifile = 0
with open(pickles[ifile], 'rb') as f:
    qa_in = pickle.load(f)[0]

In [26]:
qa_in[0]

{'Q': 'You are a helpful assistant that can analyze images.  How many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'A': {'nrows': 1, 'ncols': 2},
 'Level': 'Level 1',
 'type': 'Figure-level questions',
 'Response': {'explanation': "The figure contains two panels arranged horizontally side by side. The left panel is a histogram showing 'Progenitor' vs 'Lsb h_obs', and the right panel is a scatter plot showing 'Distances Interaction' vs 'Probed Sample Line'. This arrangement represents a single row with two columns."},
 'persona': 'You are a helpful assistant that can analyze images.',
 'context': '',
 'question': 'How many panels are in this figure?',
 'format': 'Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns.',
 'reasoning': 'In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted

Claude outputs reasoning, so we have to do a bit of cleaning from the responses:

In [27]:
print(pickles[ifile])
print('*********')
for qa_pairs in qa_in:
    print('Prompt:', qa_pairs['prompt'])
    print('  Real A:', qa_pairs['A'])
    print('Claude A:', qa_pairs['raw answer'])
    print('')

/Users/jnaiman/Dropbox/jcdl_followup/LMM_outputs/claude_haiku_thinking_maxT8000/Picture_000193_qa.pickle
*********
Prompt: I am going to show you an image. Now, how many panels are in this figure? Please format the output as a json as {"nrows":"", "ncols":""} to store the number of rows and columns. In addition to providing your answer, please provide your reasoning.  Include this as a separate JSON snippet formatted as {"explanation":""} and fill in your reasoning as a string.  Your answer should only include the answer JSON and the explanation JSON snippets, no other text.
  Real A: {'nrows': 1, 'ncols': 2}
Claude A: ```json
{
  "nrows": "1",
  "ncols": "2"
}
```

```json
{
  "explanation": "The figure contains two panels arranged horizontally side by side. The left panel is a histogram showing 'Progenitor' vs 'Lsb h_obs', and the right panel is a scatter plot showing 'Distances Interaction' vs 'Probed Sample Line'. This arrangement represents a single row with two columns."
}
```

P

In [28]:
# how many have an error in thinking token limits
for pfile in pickles[ifile]:
    with open(pickles[ifile], 'rb') as f:
        qa_in = pickle.load(f)[0]
    for qa_pairs in qa_in:
        # question_list['Error']
        if 'Error' in qa_pairs:
            import sys; sys.exit()